In [44]:
import random


def random_matrix():
    index_2d = [(i, j) for i in range(3) for j in range(3)]
    pos = random.choice(index_2d)   

    matrix = [[0 for _ in range(3)] for _ in range(3)]
    nums = 3
    while nums > 0:
        i, j = random.choice(index_2d)
        matrix[i][j] = 1
        nums -= 1
        index_2d.remove((i, j)) 
    return pos, matrix


def get_actions(i, j):
    actions = []
    if i > 0: actions.append("up")
    if i < 2: actions.append("down")
    if j > 0: actions.append("left")
    if j < 2: actions.append("right")
    return actions


class persistent:
    state = {
        "pos": (0, 0),
        "trash": 0,
        "visited": set()
    }
    action = None

    @staticmethod
    def model(pos, move):
        moves = {
            "up": (-1, 0),
            "down": (1, 0),
            "left": (0, -1),
            "right": (0, 1)
        }
        if move in moves:
            return (pos[0] + moves[move][0], pos[1] + moves[move][1])
        return pos
    
    @staticmethod
    def rules(state):
        action = []
        if state["trash"] == 1:
            action.append("suck")
        
        i, j = state["pos"]
        possible_moves = get_actions(i, j)

        unvisited_moves = []
        for move in possible_moves:
            next_pos = persistent.model((i, j), move)
            if next_pos not in state["visited"]:
                unvisited_moves.append(move)
            
        if len(unvisited_moves) > 0:
            valid_moves = unvisited_moves
        else:
            valid_moves = possible_moves

        for a in valid_moves:
            action.append(a)

        return action
    

def update_state(state, action, percept, model):
    i, j, trash = percept

    state["pos"] = (i, j)
    state["trash"] = trash
    state["visited"].add((i, j))

    return state


def rule_match(state, rules):
    return rules(state)


def action_by(rule):
    actions = []
    if rule[0] == "suck": 
        actions.append("suck")
        if len(rule) > 1:
            actions.append(random.choice(rule[1:]))
    else:
        actions.append(random.choice(rule))
    return actions


def robot_hut_bui(percept):
    
    persistent.state = update_state(persistent.state, persistent.action, percept, persistent.model)
    rule = rule_match(persistent.state, persistent.rules)
    action = action_by(rule)

    persistent.action = action

    return action


def actuator(matrix, pos, action):
    i, j = pos
    moves = {
        "up": (-1, 0),
        "down": (1, 0),
        "left": (0, -1),
        "right": (0, 1)
    }
    if action[0] == "suck":
        matrix[i][j] = 0
        move = moves[action[1]]
    else:
        move = moves[action[0]]
    pos = pos[0]+move[0], pos[1]+move[1]

    return matrix, pos


if __name__ == "__main__":
    pos, matrix = random_matrix()
    GOAL = [[0, 0, 0],
            [0, 0, 0],
            [0, 0, 0]]
    
    print(pos)
    for row in matrix: print(row)

    res = []
    history = set()
    while matrix != GOAL:
        percept = (pos[0], pos[1], matrix[pos[0]][pos[1]])

        env_state = (pos, tuple(tuple(row) for row in matrix))
        if env_state in history:
            print("Robot bị kẹt")
            break
        history.add(env_state)

        action = robot_hut_bui(percept)

        res.append("+".join(action))
        
        matrix, pos = actuator(matrix, pos, action)

    print(matrix)
    
    print(" -> ".join(res))

(0, 1)
[0, 0, 0]
[0, 1, 0]
[1, 1, 0]
[[0, 0, 0], [0, 0, 0], [0, 0, 0]]
left -> down -> down -> suck+right -> suck+up -> suck+right
